In [2]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
import fenic as fc
import os
import ast

In [ ]:
config = fc.SessionConfig(
    app_name="mcp-docs",
    semantic=fc.SemanticConfig(
        language_models={
            "mini": fc.OpenAIModelConfig(
                model_name="gpt-4o-mini",
                rpm=500,
                tpm=200_000,
            )
        }
    ),
)

# Create session
session = fc.Session.get_or_create(config)

In [265]:
def read_source_files(directory: str, extensions: list = ['.py', '.md', '.json']):
      files_data = []
      for root, dirs, files in os.walk(directory):
          for file in files:
              if any(file.endswith(ext) for ext in extensions):
                  file_path = os.path.join(root, file)
                  try:
                      with open(file_path, 'r', encoding='utf-8') as f:
                          content = f.read()
                      files_data.append({
                          'file_path': file_path,
                          'file_name': file,
                          'extension': os.path.splitext(file)[1],
                          'content': content,
                          'size': len(content)
                      })
                  except Exception as e:
                      print(f"Error reading {file_path}: {e}")
      return files_data


files_data = read_source_files("./src")

In [ ]:
raw_df = session.createDataFrame(files_data)
raw_df.show()

In [267]:
# Define the return type structure
@fc.udf(return_type=fc.StructType([
    fc.StructField("elements", fc.ArrayType(fc.StructType([
        fc.StructField("type", fc.StringType),
        fc.StructField("name", fc.StringType),
        fc.StructField("docstring", fc.StringType),
        fc.StructField("line", fc.IntegerType),
        fc.StructField("parent_class", fc.StringType)
    ]))),
    fc.StructField("parse_error", fc.StringType)
]))
def extract_python_docs(content):
    """Extract docstrings and basic info from Python code using AST."""
    try:
        tree = ast.parse(content)
        elements = []

        # Extract module docstring
        module_docstring = ast.get_docstring(tree)
        if module_docstring:
            elements.append({
                "type": "module",
                "name": "__module__",
                "docstring": module_docstring,
                "line": 1,
                "parent_class": None
            })

        # Extract classes and functions
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                elements.append({
                    "type": "function",
                    "name": node.name,
                    "docstring": ast.get_docstring(node) or "",
                    "line": node.lineno,
                    "parent_class": None
                })
            elif isinstance(node, ast.ClassDef):
                elements.append({
                    "type": "class",
                    "name": node.name,
                    "docstring": ast.get_docstring(node) or "",
                    "line": node.lineno,
                    "parent_class": None
                })

        return {
            "elements": elements,
            "parse_error": None
        }
    except SyntaxError as e:
        return {
            "elements": [],
            "parse_error": f"Syntax error: {str(e)}"
        }
         
@fc.udf(return_type=fc.StructType([
    fc.StructField("elements", fc.ArrayType(fc.StructType([
        fc.StructField("type", fc.StringType),
        fc.StructField("name", fc.StringType),
        fc.StructField("docstring", fc.StringType),
        fc.StructField("line", fc.IntegerType),
        fc.StructField("parent_class", fc.StringType)
    ]))),
    fc.StructField("parse_error", fc.StringType)
]))
def extract_python_docs_with_methods(content):
    """Extract docstrings including methods within classes."""
    try:
        tree = ast.parse(content)
        elements = []

        # Extract module docstring
        module_docstring = ast.get_docstring(tree)
        if module_docstring:
            elements.append({
                "type": "module",
                "name": "__module__",
                "docstring": module_docstring,
                "line": 1,
                "parent_class": None
            })

        # Walk only top-level nodes
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                elements.append({
                    "type": "function",
                    "name": node.name,
                    "docstring": ast.get_docstring(node) or "",
                    "line": node.lineno,
                    "parent_class": None
                })

            elif isinstance(node, ast.AsyncFunctionDef):
                elements.append({
                    "type": "async_function",
                    "name": node.name,
                    "docstring": ast.get_docstring(node) or "",
                    "line": node.lineno,
                    "parent_class": None
                })

            elif isinstance(node, ast.ClassDef):
                # Add the class itself
                elements.append({
                    "type": "class",
                    "name": node.name,
                    "docstring": ast.get_docstring(node) or "",
                    "line": node.lineno,
                    "parent_class": None
                })

                # Extract methods within the class
                for item in node.body:
                    if isinstance(item, ast.FunctionDef):
                        elements.append({
                            "type": "method",
                            "name": item.name,
                            "docstring": ast.get_docstring(item) or "",
                            "line": item.lineno,
                            "parent_class": node.name
                        })
                    elif isinstance(item, ast.AsyncFunctionDef):
                        elements.append({
                            "type": "async_method",
                            "name": item.name,
                            "docstring": ast.get_docstring(item) or "",
                            "line": item.lineno,
                            "parent_class": node.name
                        })

        return {
            "elements": elements,
            "parse_error": None
        }
    except SyntaxError as e:
        return {
            "elements": [],
            "parse_error": f"Syntax error: {str(e)}"
        }

In [ ]:
# Apply the enhanced UDF to extract documentation with methods
docs_df = raw_df.filter(
    fc.col("extension") == ".py"
).select(
    fc.col("file_path"),
    fc.col("file_name"),
    extract_python_docs_with_methods(fc.col("content")).alias("docs")
)

# Verify the extraction worked
print("Total Python files processed:", docs_df.count())

In [ ]:
# Recreate docs_df with the correct UDF
print("Recreating docs_df with the working UDF...")
docs_df_new = raw_df.filter(
    fc.col("extension") == ".py"
).select(
    fc.col("file_path"),
    fc.col("file_name"),
    extract_python_docs_with_methods(fc.col("content")).alias("docs")
)

# Create the docs DataFrame
docs_new = docs_df_new.unnest("docs").filter(fc.col("parse_error").is_null()).explode("elements").unnest("elements")

# Verify it worked
print("Checking the new extraction:")
docs_new.filter(
    (fc.col("file_name") == "session.py") &
    (fc.col("type") == "method")
).select("name", "parent_class").show(5)

print(f"\nTotal methods with parent_class: {docs_new.filter((fc.col('type') == 'method') & (fc.col('parent_class').is_not_null())).count()}")

In [270]:
docs = docs_df.unnest("docs").filter(fc.col("parse_error").is_null()).explode("elements").unnest("elements")


In [271]:
@fc.udf(return_type=fc.StringType)
def path_to_module(file_path):
      """Convert file path to Python module name."""
      # Remove ./src/ prefix
      if file_path.startswith("./src/"):
          file_path = file_path[6:]
      elif file_path.startswith("src/"):
          file_path = file_path[4:]

      # Remove .py extension
      if file_path.endswith(".py"):
          file_path = file_path[:-3]

      # Replace / with .
      return file_path.replace("/", ".")

In [272]:
cleaned_docs = docs.select(
      "*",
      path_to_module(fc.col("file_path")).alias("module_name"),
      fc.text.regexp_replace(fc.col("file_path"), r"^\./src/", "").alias("clean_path")
  )

In [273]:
# Add fully qualified names
docs_with_fqn = cleaned_docs.select(
    "*",

    fc.when(
        fc.col("type") == "module",
        fc.col("module_name")
    ).when(
        fc.col("type").is_in(["method", "async_method"]),
        fc.text.concat_ws(".", fc.col("module_name"), fc.col("parent_class"), fc.col("name"))
    ).otherwise(
        fc.text.concat_ws(".", fc.col("module_name"), fc.col("name"))
    ).alias("qualified_name_full")
)

In [274]:
  # Add more metadata
final_docs = docs_with_fqn.select(
      "*",

      # Package name (first part of module)
      fc.text.split(fc.col("module_name"), r"\.").get_item(0).alias("package"),

      # Submodule path (everything after first dot)
      # Using a combination of split and array operations
      fc.when(
          fc.col("module_name").contains("."),
          fc.text.split(fc.col("module_name"), r"\.", 1).get_item(1)
      ).otherwise(fc.lit("None")).alias("submodule"),

      # Is it a dunder method?
      fc.col("name").starts_with("__").alias("is_dunder_start"),
      fc.col("name").ends_with("__").alias("is_dunder_end"),
      # True dunder methods start AND end with __
      (fc.col("name").starts_with("__") & fc.col("name").ends_with("__")).alias("is_dunder")
  )

In [ ]:
  # Clean up - remove intermediate columns
final_docs_clean = final_docs.select(
      "file_path", "file_name", "clean_path", "module_name",
      "package", "submodule", "qualified_name_full",
      "type", "name", "docstring", "line",
      "is_dunder", "parent_class", "parse_error"
  )

# Show some statistics
print(f"Total elements extracted: {final_docs_clean.count()}")
print(f"\nBreakdown by type:")
final_docs_clean.group_by("type").agg(fc.count("*").alias("count")).show()

print(f"\nDunder methods/attributes:")
final_docs_clean.filter(fc.col("is_dunder")).select("qualified_name_full", "type").show(10)

In [ ]:
file_summaries = final_docs_clean.group_by(
      "file_path", "clean_path", "module_name", "package"
  ).agg(
      fc.count("*").alias("element_count"),
      fc.count(fc.when(fc.col("type") == "class", 1)).alias("class_count"),
      fc.count(fc.when(fc.col("type") == "function", 1)).alias("function_count"),
      fc.count(fc.when(fc.col("type") == "method", 1)).alias("method_count"),
      fc.collect_list(fc.when(fc.col("type") == "class", fc.col("name"))).alias("classes"),
      fc.collect_list(fc.when(fc.col("type") == "function", fc.col("name"))).alias("functions"),
      fc.first(fc.when(fc.col("type") == "module", fc.col("docstring"))).alias("module_docstring")
  )